# 整数规划（MILP）：一个模型一个 Notebook

## 这个模型解决什么问题
决策变量必须是整数（产量是整件、车辆是整台）或只能是 0/1（建不建这个仓、选不选这个项目）。

```
min  c'x
s.t. A_ub · x <= b_ub
     x ∈ Z（或 x ∈ {0,1}）
```

## 两种典型 0-1 模型

**① 集合覆盖 / 选址**：选最少的点（或成本最低），把所有需求点覆盖住。
```
min  Σ 固定成本_i · y_i
s.t. Σ_{i 覆盖 j} y_i >= 1    对每个需求点 j
     y_i ∈ {0,1}
```

**② 0-1 背包**：容量有限，选哪些物品使总价值最大。
```
max  Σ 价值_i · y_i
s.t. Σ 重量_i · y_i <= 容量
```

## 两种数据输入方式
| 方式 | 用法 |
| --- | --- |
| 原始表（选址：候选仓 + 覆盖需求点字符串） | `build_cover_from_table` |
| 已处理矩阵（背包：价值/重量两列） | `solve_milp` 直接给系数 |

## 目录
0 环境 / 1 配置 / 2 数据接口 / 3 建模与求解 / 4 原始表示例（选址）/ 5 矩阵示例（背包）/ 6 导出

> 依赖 `scipy >= 1.9`（`scipy.optimize.milp` 从 1.9 起提供）。


In [1]:
# -*- coding: utf-8 -*-
# =====================================================================
# 0. 环境与路径（每个 Notebook 都带这一段，保证单独打开就能跑）
# =====================================================================
# 依赖：numpy、pandas、scipy（>=1.9，用到 milp）
# 可选：matplotlib（画图）、openpyxl（导出 xlsx）
# 缺哪个就装：pip install numpy pandas scipy matplotlib openpyxl
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import optimize as sp_opt
from scipy.optimize import Bounds, LinearConstraint

try:
    import matplotlib
    import matplotlib.pyplot as plt
    # 不设中文字体，图里的标题和标签会变成方块
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
    HAS_MPL = True
except ImportError:
    HAS_MPL = False


def find_root(marker="导入数据"):
    """向上逐级找含 导入数据/ 的目录。

    Notebook 放在 优化类模型/ 下也能找到项目根，换电脑、换目录都不用改死路径。
    """
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).is_dir():
            return p
    return Path.cwd()


ROOT = find_root()
DATA_DIR = ROOT / "导入数据"
OUT_DIR = ROOT / "输出数据"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("项目根目录：", ROOT)
print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", sp_opt.__name__ and __import__("scipy").__version__)


项目根目录： D:\git项目
numpy 2.4.6 | pandas 3.1.0 | scipy 1.17.1


In [2]:
# =====================================================================
# 1. 配置区
# =====================================================================
CFG = dict(
    data_path=DATA_DIR / "优化_选址候选.csv",   # 选址候选表
    sheet_name=0,
    cover_sep=";",          # 覆盖需求点列里的分隔符
    tag="选址规划",
    make_plots=True,
    save_excel=True,
    time_limit=30,          # 整数规划求解时间上限（秒）
)
print("配置：", CFG)


配置： {'data_path': WindowsPath('D:/git项目/导入数据/优化_选址候选.csv'), 'sheet_name': 0, 'cover_sep': ';', 'tag': '选址规划', 'make_plots': True, 'save_excel': True, 'time_limit': 30}


## 2. 数据接口


In [3]:
# =====================================================================
# 2. 数据接口：读文件 → 清洗 → 变成模型能吃的表
# =====================================================================
# 这一格是「原始数据直接放进来」的关键：不用你手动去空格、转数字、去重。
# 如果数据已经处理好，可以跳过清洗，直接用后面的 load_matrix 读矩阵。
#
# ⚠️ 999 是否算缺失取决于题目：若它是真实观测值，把它从 NA_MARKERS 里删掉。
NA_MARKERS = ("", "NA", "N/A", "na", "缺测", "未填", "无", "999", 999, "9999", 9999)


def load_table(data, sheet_name=0, encoding="utf-8-sig"):
    """读 csv / xlsx，也接受现成的 DataFrame。

    - csv 先按 utf-8-sig 读（吃掉 Excel 导出的 BOM），失败再退回 gbk
    - 列名首尾空格自动去掉（" 单耗" → "单耗"）
    """
    if isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        p = Path(data)
        if not p.exists():
            raise FileNotFoundError(f"找不到数据文件：{p}")
        if p.suffix.lower() in (".xlsx", ".xls"):
            df = pd.read_excel(p, sheet_name=sheet_name)
        else:
            try:
                df = pd.read_csv(p, encoding=encoding)
            except UnicodeDecodeError:
                df = pd.read_csv(p, encoding="gbk")
    df.columns = [str(c).strip() for c in df.columns]
    return df


def clean_frame(df, na_markers=NA_MARKERS, drop_missing_rate=50.0,
                fill_num="median", diagnostics=None):
    """把原始附件洗成能建模的表，并把每一步记进 diagnostics。

    顺序有因果，别随意调换：
    ① 哨兵值 → NaN       999 / 缺测 / 未填 统一当缺失
    ② 文本列去空格       否则 "东部" 与 " 东部" 会被当成两个类别
    ③ 文本数字转数值     " 12.5 " 这种带空格的数字
    ④ 删完全重复行
    ⑤ 高缺失列删列       删列不删行，保住样本量
    ⑥ 缺失填补           数值列中位数/均值，文本列众数

    参数
    ----
    diagnostics : 传一个 list 进来，会往里追加每一步的说明，最后可以导出给论文用
    """
    diag = diagnostics if diagnostics is not None else []
    d = df.copy()
    d.columns = [str(c).strip() for c in d.columns]

    before = int(d.isna().sum().sum())
    d = d.replace(list(na_markers), np.nan)
    diag.append({"步骤": "哨兵值转缺失",
                 "说明": f"新增缺失 {int(d.isna().sum().sum()) - before} 个"})

    obj_cols = [c for c in d.columns
                if pd.api.types.is_string_dtype(d[c]) or pd.api.types.is_object_dtype(d[c])]
    for c in obj_cols:
        d[c] = d[c].astype("string").str.strip()
    if obj_cols:
        diag.append({"步骤": "文本列去空格", "说明": f"处理 {len(obj_cols)} 列"})

    num_cols, cat_cols = [], []
    for c in d.columns:
        s = d[c]
        if pd.api.types.is_numeric_dtype(s):
            num_cols.append(c)
            continue
        conv = pd.to_numeric(s.astype(str).str.strip(), errors="coerce")
        if conv.notna().sum() / max(s.notna().sum(), 1) > 0.9:
            d[c] = conv
            num_cols.append(c)
        else:
            cat_cols.append(c)
    diag.append({"步骤": "列类型识别",
                 "说明": f"数值列 {len(num_cols)} 个、类别列 {len(cat_cols)} 个"})

    n_dup = int(d.duplicated().sum())
    d = d.drop_duplicates(keep="first").reset_index(drop=True)
    diag.append({"步骤": "去完全重复", "说明": f"删除 {n_dup} 行"})

    rate = d.isna().mean() * 100
    drop = list(rate[rate >= drop_missing_rate].index)
    if drop:
        d = d.drop(columns=drop)
        diag.append({"步骤": "删除高缺失列",
                     "说明": f"删除 {drop}（缺失率 ≥ {drop_missing_rate}%），删列不删行"})

    filled = []
    for c in num_cols:
        if c in d.columns and d[c].isna().any():
            fill = {"median": d[c].median(), "mean": d[c].mean(), "zero": 0}[fill_num]
            d[c] = d[c].fillna(fill)
            filled.append(c)
    for c in cat_cols:
        if c in d.columns and d[c].isna().any():
            m = d[c].mode(dropna=True)
            d[c] = d[c].fillna(m.iat[0] if len(m) else "未知")
            filled.append(c)
    if filled:
        diag.append({"步骤": "缺失填补", "说明": f"填补 {filled}，数值方式 {fill_num}"})

    diag.append({"步骤": "清洗完成", "说明": f"{d.shape[0]} 行 × {d.shape[1]} 列"})
    return d


def pivot_long(df, index, columns, values, agg="sum"):
    """长表转宽表：产品-资源-单耗 三列 → 产品 × 资源 矩阵。

    真实附件常见这种长表；同一 (产品, 资源) 出现多次时按 agg 聚合。
    """
    wide = df.pivot_table(index=index, columns=columns, values=values, aggfunc=agg).reset_index()
    wide.columns = [str(c) for c in wide.columns]
    return wide


def load_matrix(data, sheet_name=0):
    """读成本矩阵 / 距离矩阵：第一列当行名，其余列转成数值。"""
    df = load_table(data, sheet_name)
    if df.shape[1] > 1 and not pd.api.types.is_numeric_dtype(df.iloc[:, 0]):
        df = df.set_index(df.columns[0])
    df.index = [str(i) for i in df.index]
    df.columns = [str(c) for c in df.columns]
    return df.astype(float)


## 3. 建模与求解

两个 0-1 模型：集合覆盖（选址）与背包。`solve_milp` 是通用入口，换模型只需换系数。


In [4]:
# =====================================================================
# 3. 建模与求解
# =====================================================================
def build_cover_from_table(df, var_col="候选仓", cost_col="固定成本",
                           cover_col="覆盖需求点", sep=";", demands=None):
    """集合覆盖：候选表 → 0-1 规划系数。

    表结构：一行一个候选点，覆盖关系写成 "D1;D2;D3"。
    返回 dict，包含变量名、每个需求点被谁覆盖的 0/1 矩阵、成本、诊断信息。
    """
    d = df.copy()
    d.columns = [str(c).strip() for c in d.columns]
    names = d[var_col].astype(str).tolist()
    n = len(names)

    # 把 "D1;D2;D3" 拆成集合；顺带兼容中文分号
    cover_sets = []
    for s in d[cover_col]:
        parts = str(s).replace("，", sep).replace("；", sep).split(sep)
        cover_sets.append({x.strip() for x in parts if x.strip()})
    points = list(demands) if demands else sorted({x for s in cover_sets for x in s})

    # 覆盖矩阵：行=需求点，列=候选点，1 表示能覆盖
    A = np.zeros((len(points), n))
    for j, p in enumerate(points):
        for i, s in enumerate(cover_sets):
            A[j, i] = 1.0 if p in s else 0.0

    empty = [points[j] for j in range(len(points)) if A[j].sum() == 0]
    if empty:
        raise ValueError(f"需求点 {empty} 没有任何候选点能覆盖，问题无可行解——"
                         "要么加候选点，要么放宽覆盖要求")

    cost = (pd.to_numeric(d[cost_col], errors="coerce").fillna(0).to_numpy()
            if cost_col in d.columns else np.ones(n))
    diag = [f"{n} 个候选点、{len(points)} 个需求点",
            f"每个需求点至少有 {int(A.sum(axis=1).min())} 个候选点可覆盖"]
    # 约束写成 -A · y <= -1，等价于 A · y >= 1
    return dict(c=cost, A_ub=-A, b_ub=-np.ones(len(points)),
                bounds=[(0, 1)] * n, integrality=np.ones(n, dtype=int),
                var_names=names, points=points, cover_matrix=A, diag=diag)


def solve_milp(c, A_ub=None, b_ub=None, bounds=None, integrality=None,
               maximize=False, var_names=None, time_limit=30):
    """调用 scipy.optimize.milp 求解混合整数规划。

    注意：milp 的 bounds 必须传 Bounds 对象，传 [(0,1), ...] 会报
    "Bounds.__init__() takes from 1 to 4 positional arguments"。
    这里已经替你转换好了。
    """
    t0 = time.time()
    c = np.asarray(c, dtype=float).ravel()
    n = len(c)
    bounds = list(bounds) if bounds is not None else [(0, None)] * n
    integ = (np.asarray(integrality, dtype=int) if integrality is not None
             else np.zeros(n, dtype=int))
    lb = np.array([b[0] if b[0] is not None else -np.inf for b in bounds], dtype=float)
    ub = np.array([b[1] if b[1] is not None else np.inf for b in bounds], dtype=float)

    cons = []
    if A_ub is not None:
        cons.append(LinearConstraint(np.atleast_2d(np.asarray(A_ub, float)),
                                     -np.inf, np.asarray(b_ub, float)))

    r = sp_opt.milp(-c if maximize else c, integrality=integ,
                    bounds=Bounds(lb, ub), constraints=cons or None,
                    options={"time_limit": time_limit})
    x = r.x if r.x is not None else np.full(n, np.nan)
    obj = float(-r.fun if maximize else r.fun) if r.fun is not None else np.nan
    return dict(x=x, objective=obj, ok=bool(r.success), status=int(r.status),
                message=str(r.message),
                var_names=list(var_names) if var_names is not None else [f"x{i}" for i in range(n)],
                maximize=maximize, elapsed=time.time() - t0)


def knapsack_tables(res, values, weights, capacity, item_names):
    """背包结果整理：选了哪些、总价值、总重量、剩余容量。"""
    chosen = [item_names[i] for i in range(len(res["x"])) if res["x"][i] > 0.5]
    t_sel = pd.DataFrame({
        "物品": item_names,
        "是否选中": ["是" if v > 0.5 else "否" for v in res["x"]],
        "价值": values, "重量": weights,
    })
    used = float(np.dot(np.asarray(res["x"], float), weights))
    t_sum = pd.DataFrame([{
        "选中件数": len(chosen), "总价值": round(res["objective"], 4),
        "总重量": round(used, 4), "容量": capacity,
        "剩余容量": round(capacity - used, 4),
    }])
    return t_sel, t_sum, chosen


## 4. 示例 A：候选表 → 集合覆盖


In [5]:
# =====================================================================
# 4. 示例 A：候选表 → 集合覆盖（原始表模式）
# =====================================================================
raw = load_table(CFG["data_path"])
print("候选表：", raw.shape)
print(raw.to_string(index=False))

model = build_cover_from_table(raw, var_col="候选仓",
                               cost_col="固定成本(万元)",
                               cover_col="覆盖需求点",
                               sep=CFG["cover_sep"])
print("\n--- 建模信息 ---")
for line in model["diag"]:
    print(" ·", line)

res_a = solve_milp(**{k: model[k] for k in
                      ("c", "A_ub", "b_ub", "bounds", "integrality", "var_names")},
                   time_limit=CFG["time_limit"])

selected = [n for n, v in zip(model["var_names"], res_a["x"]) if v > 0.5]
t_choice = pd.DataFrame({"候选点": model["var_names"],
                         "是否选中": ["✔" if v > 0.5 else "" for v in res_a["x"]],
                         "固定成本": model["c"]})
# 覆盖核对：每个需求点被哪些选中的点覆盖
rows = []
for j, p in enumerate(model["points"]):
    covered_by = [model["var_names"][i] for i in range(len(model["var_names"]))
                  if model["cover_matrix"][j, i] > 0 and res_a["x"][i] > 0.5]
    rows.append({"需求点": p, "被选中点覆盖": "、".join(covered_by) or "❌ 未覆盖"})
t_cover = pd.DataFrame(rows)

print(f"\n=== 选址方案 ===\n选中 {len(selected)} 个：{selected}")
print(f"总固定成本 {res_a['objective']:.2f} 万元\n")
print(t_choice.to_string(index=False))
print("\n=== 覆盖核对 ===")
print(t_cover.to_string(index=False))
print("\n全覆盖：", bool((t_cover["被选中点覆盖"] != "❌ 未覆盖").all()))


候选表： (6, 4)
 候选仓  固定成本(万元)  容量(吨/日)    覆盖需求点
候选仓1       120      800 D1;D2;D3
候选仓2        95      600    D2;D4
候选仓3       140      900 D1;D5;D6
候选仓4        80      500 D3;D4;D7
候选仓5       110      700 D5;D6;D7
候选仓6       100      650    D8;D1

--- 建模信息 ---
 · 6 个候选点、8 个需求点
 · 每个需求点至少有 1 个候选点可覆盖

=== 选址方案 ===
选中 4 个：['候选仓2', '候选仓4', '候选仓5', '候选仓6']
总固定成本 385.00 万元

 候选点 是否选中  固定成本
候选仓1        120
候选仓2    ✔    95
候选仓3        140
候选仓4    ✔    80
候选仓5    ✔   110
候选仓6    ✔   100

=== 覆盖核对 ===
需求点    被选中点覆盖
 D1      候选仓6
 D2      候选仓2
 D3      候选仓4
 D4 候选仓2、候选仓4
 D5      候选仓5
 D6      候选仓5
 D7 候选仓4、候选仓5
 D8      候选仓6

全覆盖： True


## 5. 示例 B：0-1 背包（矩阵）


In [6]:
# =====================================================================
# 5. 示例 B：0-1 背包（已处理矩阵模式）
# =====================================================================
# 系数已经算好，直接给价值和重量，不用建表。
item_names = ["帐篷", "睡袋", "炉具", "水", "食物", "急救包"]
values  = [60, 45, 30, 50, 80, 25]      # 价值
weights = [8.0, 5.0, 3.0, 6.0, 7.0, 2.0]  # 重量(kg)
capacity = 20.0

res_b = solve_milp(c=values, A_ub=[weights], b_ub=[capacity],
                   bounds=[(0, 1)] * len(values),
                   integrality=np.ones(len(values), dtype=int),
                   maximize=True, var_names=item_names)
t_sel, t_sum, chosen_b = knapsack_tables(res_b, values, weights, capacity, item_names)
print("=== 背包方案（矩阵模式）===")
print(t_sel.to_string(index=False))
print("\n", t_sum.to_string(index=False))
print("\n选中：", chosen_b)


=== 背包方案（矩阵模式）===
 物品 是否选中  价值  重量
 帐篷    否  60 8.0
 睡袋    是  45 5.0
 炉具    否  30 3.0
  水    是  50 6.0
 食物    是  80 7.0
急救包    是  25 2.0

  选中件数   总价值  总重量   容量  剩余容量
    4 200.0 20.0 20.0   0.0

选中： ['睡袋', '水', '食物', '急救包']


## 6. 结果导出


In [7]:
# =====================================================================
# 结果导出：CSV + 多 Sheet Excel + 图
# =====================================================================
def export_result(tables, cfg, fig=None, fig_name="结果图"):
    """把结果写成文件，返回写出的路径。

    tables : {Sheet 名: DataFrame}，第一张建议放"摘要"
    cfg    : 配置字典，用 tag / make_plots / save_excel 三个键
    fig    : 传了 matplotlib 图对象就保存；cfg["make_plots"]=False 时连图都不生成
    """
    d = OUT_DIR / f"优化_{cfg['tag']}"
    d.mkdir(parents=True, exist_ok=True)
    written = {}

    first = next(iter(tables.values()))
    p_csv = d / f"{cfg['tag']}_主表.csv"
    first.to_csv(p_csv, index=False, encoding="utf-8-sig")
    written["主表"] = p_csv

    if cfg.get("save_excel", True):
        p_xlsx = d / f"{cfg['tag']}_结果汇总.xlsx"
        with pd.ExcelWriter(p_xlsx, engine="openpyxl") as w:
            for name, frame in tables.items():
                frame.to_excel(w, sheet_name=str(name)[:28], index=False)
        written["汇总"] = p_xlsx
        print(f"[表格] {p_xlsx.name}（{len(tables)} 个 Sheet）")

    if fig is not None and cfg.get("make_plots", True) and HAS_MPL:
        p_png = d / f"{cfg['tag']}_{fig_name}.png"
        fig.savefig(p_png, dpi=150, bbox_inches="tight")
        if "agg" not in matplotlib.get_backend().lower():   # 无界面后端不弹窗，也不报警告
            plt.show()
        plt.close(fig)          # 关掉画布：反复运行不会在内存里堆图
        written["图片"] = p_png
        print(f"[图片] {p_png.name}")

    print("[输出] ", d)
    return written


In [8]:
# =====================================================================
# 6. 结果导出
# =====================================================================
fig = None
if CFG["make_plots"] and HAS_MPL:
    fig, ax = plt.subplots(figsize=(6.6, 4))
    picked = [n for n, v in zip(model["var_names"], res_a["x"]) if v > 0.5]
    colors = ["#4a9d6a" if n in picked else "#c9ccd1" for n in model["var_names"]]
    ax.bar(model["var_names"], model["c"], color=colors)
    ax.set_ylabel("固定成本（万元）")
    ax.set_title(f"选址结果：选中 {len(picked)} 个，总成本 {res_a['objective']:.0f}（绿色=选中）")
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
    fig.tight_layout()

written = export_result(
    {"结果摘要": pd.DataFrame([{"选点数": len(selected), "总成本": res_a["objective"],
                              "状态": "最优" if res_a["ok"] else "未最优"}]),
     "选址方案": t_choice, "覆盖核对": t_cover, "背包方案": t_sel},
    CFG, fig=fig, fig_name="选址方案")
print(written)


[表格] 选址规划_结果汇总.xlsx（4 个 Sheet）
[图片] 选址规划_选址方案.png
[输出]  D:\git项目\输出数据\优化_选址规划
{'主表': WindowsPath('D:/git项目/输出数据/优化_选址规划/选址规划_主表.csv'), '汇总': WindowsPath('D:/git项目/输出数据/优化_选址规划/选址规划_结果汇总.xlsx'), '图片': WindowsPath('D:/git项目/输出数据/优化_选址规划/选址规划_选址方案.png')}


换题目时按三步走：

1. 数据放进 `导入数据/`（csv / xlsx 都行）；
2. 改本 Notebook 第 1 个代码单元里的 `CFG`：文件名、列名映射、方向、输出名；
3. `Run All`，结果写到 `输出数据/优化_{tag}/`，同名覆盖，不会越跑越多。

**两种输入方式怎么选**

- 数据还脏（有空行、重复、999、文本数字、列名带空格）→ 走「原始数据」示例，让 `clean_frame` 处理；
- 数据已经清洗/标准化完（比如上一节 `数据学习.ipynb` 输出的 `final_cleaned.csv`）→ 走「已处理数据」示例，直接给矩阵或宽表。

不要混：**同一次建模只用一种入口**，否则等于把同一份数据预处理了两遍。
